# 09 Advanced Challenge - PCA and Heatmap for Toy Gene Expression in Python

## Biochemistry question

In this toy gene expression matrix, do control, low-treatment, and high-treatment samples show visible expression patterns?


In [1]:
import plotly.io as pio
pio.renderers.default = "iframe"


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/gene_expression_matrix_toy.csv")
df.head()

,gene,control_1,control_2,control_3,treat_low_1,treat_low_2,treat_low_3,treat_high_1,treat_high_2,treat_high_3
0,EGFR,10.2,9.8,10.5,12.1,12.4,11.9,16.5,17.1,15.9
1,TP53,8.1,8.4,8.0,9.5,9.8,9.4,12.2,11.8,12.5
2,MYC,15.0,14.8,15.3,17.5,17.8,17.2,21.4,22.1,20.9
3,GAPDH,30.2,30.1,29.8,30.4,30.2,30.1,30.5,30.0,30.3
4,ACTB,25.1,25.4,24.9,25.0,25.2,25.3,25.2,25.0,25.5


In [3]:
expr = df.set_index("gene")
# z-score each gene across samples for heatmap visualization
expr_z = expr.sub(expr.mean(axis=1), axis=0).div(expr.std(axis=1), axis=0)
expr_z.head()

,control_1,control_2,control_3,treat_low_1,treat_low_2,treat_low_3,treat_high_1,treat_high_2,treat_high_3
gene,,,,,,,,,
EGFR,-0.965324,-1.106591,-0.859374,-0.294306,-0.188356,-0.364940,1.259630,1.471531,1.047730
TP53,-1.053002,-0.883769,-1.109413,-0.263250,-0.094018,-0.319661,1.259841,1.034198,1.429074
MYC,-1.058677,-1.129255,-0.952809,-0.176446,-0.070578,-0.282314,1.199834,1.446859,1.023388
GAPDH,0.105409,-0.368932,-1.791957,1.054093,0.105409,-0.368932,1.528434,-0.843274,0.579751
ACTB,-0.391618,1.118908,-1.398636,-0.895127,0.111891,0.615400,0.111891,-0.895127,1.622417


In [4]:
fig = px.imshow(
    expr_z,
    aspect="auto",
    title="Toy Gene Expression Heatmap (gene-wise z-score)",
    labels=dict(x="Sample", y="Gene", color="z-score")
)
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


In [5]:
# PCA using NumPy SVD.
# Samples should be rows and genes should be columns.
X = expr.T
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
pc_scores = U[:, :2] * S[:2]

pca_df = pd.DataFrame({
    "sample": X.index,
    "PC1": pc_scores[:, 0],
    "PC2": pc_scores[:, 1],
})
pca_df["group"] = pca_df["sample"].str.extract(r"^(control|treat_low|treat_high)")
pca_df

,sample,PC1,PC2,group
0,control_1,-4.397849,0.205717,control
1,control_2,-4.281932,1.189000,control
2,control_3,-4.643368,-1.706986,control
3,treat_low_1,-0.300801,-0.049401,treat_low
4,treat_low_2,-0.142132,0.173555,treat_low
5,treat_low_3,-0.542613,0.393810,treat_low
6,treat_high_1,4.902548,0.417091,treat_high
7,treat_high_2,4.973057,-1.834703,treat_high
8,treat_high_3,4.433090,1.211916,treat_high


In [6]:
fig2 = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="group",
    text="sample",
    title="Toy PCA-style Sample Map"
)
# If this chart does not render in Jupyter, try: fig2.show(renderer="browser")
fig2.show(renderer="iframe")


## Interpretation Questions

1. Do control, low-treatment, and high-treatment samples separate visually?
2. What does a heatmap show?
3. What does PCA help visualize?
4. Why is this still not a full RNA-seq workflow?

## Limitations

- This is a toy synthetic gene expression matrix.
- Visual clustering does not prove biological mechanism or diagnostic meaning.
- A real workflow would need appropriate normalization, QC, statistical testing, and validation.
